In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import matplotlib.cm as cm
import datetime
import scipy.stats
#import plotly.offline as pyo
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.colors as cl
import matplotlib.dates as mdates
import matplotlib
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS
import seaborn as sns
import netCDF4 as nc
import xarray as xr
#import cartopy.crs as ccrs
#from cartopy import feature as cfa
import os
import plotly.io as pio
pio.renderers.default = 'notebook'

In [ ]:
def decimal_year_to_datetime(decimal_year):
    year = decimal_year.astype(int)
    decimal_part = decimal_year - year
    start_of_year = pd.to_datetime([f'{y}-01-01' for y in year])
    seconds_in_year = (pd.to_datetime([f'{y+1}-01-01' for y in year]) - start_of_year).total_seconds()
    seconds = decimal_part * seconds_in_year
    return start_of_year + pd.to_timedelta(seconds, unit='s')

def datetime_to_decimal_year(datetime_):
    year = datetime_.year
    days = (datetime_.date() - datetime.date(year, 1,1)).days
    res = year + days/365.25
    return res


def FFT_res(x, si):
    n = len(x)
    x_FFT_shifted = np.fft.fftshift(np.fft.fft(x))
    x_FFT_freq_shifted = np.fft.fftshift(np.fft.fftfreq(n, d=si))
    return x_FFT_freq_shifted, x_FFT_shifted


def fft_noramlize(fft_res):
    n = len(fft_res)
    norm_fft = np.abs(fft_res)/(np.max(np.abs(fft_res)))
    return norm_fft

def line_fit(X,Y):
    P = np.polyfit(X,Y,1)
    dx = np.max(X) - np.min(X)
    X_new = np.array([np.min(X)-dx*0.05, np.max(X)+dx*0.05])
    Y_new = np.polyval(P, X_new)
    return X_new, Y_new, P

In [ ]:
%run 20260223_read_data.py
df_sorted

In [ ]:
climate = pd.read_csv('../../2026-01-15_Climate-Model/analysis/climate_df.csv')
climate['datetime'] = pd.to_datetime(climate['datetime'])
climate['datetime_month'] = climate['datetime'].dt.month
climate_monthly = climate[['SST','SSS','age_absolute', 'synthetic_d18o', 'synthetic_d18o_2','datetime_month']].groupby('datetime_month').mean()
climate_monthly

In [ ]:
tide = pd.read_csv('../../2025-10-10_Fred-TideGauge/memo_edited.csv')
tide['age_dt'] = decimal_year_to_datetime(tide['Date'])#.dt.date
tide = tide.set_index('age_dt')
tide

In [ ]:
enso_phase = pd.read_csv('../../ENSO_Phase_Classification.csv')
enso_phase.columns=['Year','Phase']
enso_phase['Date'] = decimal_year_to_datetime(enso_phase['Year'])
#enso_phase.set_index('Date', inplace=True)
enso_phase['Phase'] = enso_phase['Phase'].str.replace('Warm Phase', 'W').str.replace('Cold Phase', 'C').str.replace('Neutral Phase','N')


enso = pd.read_csv("../../ENSO_Time_Series.csv", parse_dates=[0])
#enso = enso.set_index("Date")
enso['day_of_year'] = enso['Date'].dt.dayofyear
enso['is_leap'] = enso['Date'].dt.is_leap_year
enso['total_days'] = np.where(enso['is_leap'], 366, 365)
enso['decimal_year'] = enso['Date'].dt.year + (enso['day_of_year'] - 1) / enso['total_days']


margin = 0.5
enso['phase'] = 'N'
enso.loc[enso['ENSO']>=margin,'phase'] = "C"
enso.loc[enso['ENSO']<=-margin,'phase'] = "W"
enso['phase_group'] = (enso['phase'] != enso['phase'].shift()).cumsum()
blocks = enso.groupby('phase_group').agg(
    start=('Date', 'first'),
    end=('Date', 'last'),
    value=('phase', 'first')
).reset_index(drop=True)

ENSO_match = np.array([[1982.2, 1984.1],
                       [1986.75, 1988.3],
                       [1992.1, 1992.96],
                       [1997.2, 1998.8]])

enso_fft_freq, enso_fft = FFT_res(enso['ENSO'], 30)


enso


In [ ]:
# Ensure df_resampled is sorted by date
df_ = df_resampled[['d13c', 'd13c_no_tectonic','d18o','d18o_no_tectonic']].copy()
df_ = df_.sort_index()

# Target dates to align with
target_dates = pd.to_datetime(enso.loc[:685, 'Date'])

# Interpolate to get values at those dates
aligned = df_.reindex(df_.index.union(target_dates)).interpolate(method='time').loc[target_dates]
aligned['age_absolute'] = np.array([datetime_to_decimal_year(x) for x in pd.to_datetime(aligned.index)])
aligned['ENSO'] = enso.loc[:685, 'ENSO'].values
aligned_tidegauge = tide.reindex(tide.index.union(target_dates)).interpolate(method='time').loc[target_dates]
aligned = aligned.join(aligned_tidegauge['Sea Level'].dropna())
aligned_climate = climate.set_index('datetime')
aligned_climate = aligned_climate.reindex(aligned_climate.index.union(target_dates)).interpolate(method='time').loc[target_dates]
aligned = aligned.join(aligned_climate[['SST','SSS', 'synthetic_d18o']].dropna())

aligned.loc['2005-01-01':'2008-01-01']

In [ ]:
sampling_rates = df[~df['age1'].isna()].groupby('path_name')['path_num'].diff().dropna()

In [ ]:
dt = np.diff(df_sorted['age_final'].values)
plt.hist(dt * 12, bins=50)  # in months
plt.xlabel('Sampling interval (months)')

In [ ]:
df['age_final']

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(12,4), width_ratios=(2,1), sharey=True, gridspec_kw={'wspace':0})

ax = axes[0]
ax.plot(sampling_rates, '.-')

ax = axes[1]

mu = np.mean(sampling_rates)
sigma = np.std(sampling_rates)
bin_count = 16

count, bins, ignored = ax.hist(sampling_rates, bins=np.arange(1,bin_count), rwidth=0.9, align='left',orientation='horizontal' ,density=True)
ax.plot(1/(sigma*np.sqrt(2*np.pi))*np.exp(-(bins-mu)**2 / (2*sigma**2)),bins,linewidth=2, color='r')
ax.set_yticks(np.arange(1,bin_count+1))
ax.set_ylim(1,bin_count+1)


#fig.suptitle('Sampling Rate Distribution')

axes[1].set_title('Sampling Rate Distribution')
axes[0].set_title('Sampling Rate over Time')
axes[0].set_ylabel('# of Sample per Year')
axes[0].set_xlabel('Sample Index ~ Time')
ax.set_xlabel('Probability')

#fig.savefig('sampling_rate_distribution.pdf')

In [ ]:

growth_rates = (sampling_rates*1.5)/10

# Drill Samples were 1.5 mm ?????

#####################################

fig, axes = plt.subplots(1,2, figsize=(12,4), width_ratios=(2,1), sharey=True, gridspec_kw={'wspace':0})

ax = axes[0]
ax.plot(growth_rates, '.-')
ax.grid()

ax = axes[1]

mu = np.mean(growth_rates)
sigma = np.std(growth_rates)
bin_count = 22

count, bins, ignored = ax.hist(growth_rates, bins=np.arange(1,bin_count)/10, rwidth=0.7, align='left',orientation='horizontal' ,density=True)
ax.plot(1/(sigma*np.sqrt(2*np.pi))*np.exp(-(bins-mu)**2 / (2*sigma**2)),bins,linewidth=2, color='r')
ax.set_yticks(np.arange(1,bin_count+1,2)/10)
ax.set_ylim(1/10,bin_count/10+1/10)
ax.grid()

ax.yaxis.tick_right()
ax.yaxis.set_label_position("right")
ax.set_ylabel("Growth Rate [cm/a]")


#fig.suptitle('Sampling Rate Distribution')

axes[1].set_title('Growth Rate Distribution')
axes[0].set_title('Growth Rate over Sample Age')
axes[0].set_ylabel('Growth Rate [cm/a]')
axes[0].set_xlabel('Sample Index ~ Time')
ax.set_xlabel('Probability')


fig.savefig('sampling_rate_distribution.pdf')

In [ ]:
from astropy.timeseries import LombScargle
import numpy as np

# Use detrended signal
data = df_sorted[['age_final', 'd13c_no_tectonic']].dropna()

ls = LombScargle(data['age_final'], data['d13c_no_tectonic'])
frequency_notec , power_notec = ls.autopower(minimum_frequency=0.001,  # periods < 2 years
                                 maximum_frequency=12,   # periods > 1 month
                                 samples_per_peak=10)

period_notec = 1 / frequency_notec



In [ ]:
fig, ax = plt.subplots(1, figsize=(12, 7))

ax.plot(period_notec, power_notec)
ax.axvline(1.0, color='r', linestyle='--', alpha=0.4,label='1 year')
ax.set_xlabel('Period (years)')
ax.set_xlim(0.5, 7)

ax.legend()

In [ ]:
# Use detrended signal
data = df_sorted[['age_final', 'd13c_no_tectonic2']].dropna()

ls = LombScargle(data['age_final'], data['d13c_no_tectonic2'])
frequency_notec2, power_notec2 = ls.autopower(minimum_frequency=0.001,  # periods < 2 years
                                 maximum_frequency=12,   # periods > 1 month
                                 samples_per_peak=10)

period_notec2 = 1 / frequency_notec2

fig, ax = plt.subplots(1, figsize=(12, 7))

ax.plot(period_notec2, power_notec2)
ax.axvline(1.0, color='r', linestyle='--', alpha=0.4,label='1 year')
ax.set_xlabel('Period (years)')
ax.set_xlim(0.5, 7)

ax.legend()


In [ ]:
# Use detrended signal
data = df_sorted[['age_final', 'd13c']].dropna()

ls = LombScargle(data['age_final'], data['d13c'])
frequency, power = ls.autopower(minimum_frequency=0.001,  # periods < 2 years
                                 maximum_frequency=12,   # periods > 1 month
                                 samples_per_peak=10)

period = 1 / frequency



In [ ]:
data = df_sorted[['age_final', 'd13c']].dropna()

ls = LombScargle(data['age_final'], data['d13c'])
frequency, power = ls.autopower(minimum_frequency=0.001,  # periods < 2 years
                                 maximum_frequency=12,   # periods > 1 month
                                 samples_per_peak=10)
period = 1 / frequency

data = df_sorted[['age_final', 'd13c_no_tectonic']].dropna()

ls = LombScargle(data['age_final'], data['d13c_no_tectonic'])
frequency_notec2, power_notec2 = ls.autopower(minimum_frequency=0.001,  # periods < 2 years
                                 maximum_frequency=12,   # periods > 1 month
                                 samples_per_peak=10)
period_notec = 1 / frequency_notec



data = df_sorted[['age_final', 'd13c_no_tectonic2']].dropna()

ls = LombScargle(data['age_final'], data['d13c_no_tectonic2'])
frequency_notec2, power_notec2 = ls.autopower(minimum_frequency=0.001,  # periods < 2 years
                                 maximum_frequency=12,   # periods > 1 month
                                 samples_per_peak=10)
period_notec2 = 1 / frequency_notec2



In [ ]:
fig, ax = plt.subplots(1, figsize=(8, 5))


greens = cm.get_cmap('Greens')

colors = [
    greens(0.35),  # light green (raw)
    greens(0.65),  # medium green (36m)
    greens(0.9)    # dark green (72m)
]

ax.plot(period, power,color=colors[0], label='raw')
ax.plot(period_notec, power_notec,color=colors[1], label='36 month filter')
ax.plot(period_notec2, power_notec2,color=colors[2], label='72 month filter')

ax.axvline(1.0, color='r', linestyle='--', alpha=0.4,label='1 year')
ax.axvline(3.617, color='g', linestyle='--', alpha=0.4,label='3.6 year')
ax.set_xlabel('Period (years)')
ax.set_ylabel('Spectral Amplitude')
ax.set_title(r'$\delta^{13}$C Spectrum')
ax.set_xlim(0.5, 7)
ax.set_ylim(0,0.04)

ax.legend(loc='upper right')

fig.savefig('lombscargle_raw-filter_d13c.pdf')

In [ ]:
data = df_sorted[['age_final', 'd18o']].dropna()

ls = LombScargle(data['age_final'], data['d18o'])
frequency, power = ls.autopower(minimum_frequency=0.001,  # periods < 2 years
                                 maximum_frequency=12,   # periods > 1 month
                                 samples_per_peak=10)
period = 1 / frequency

data = df_sorted[['age_final', 'd18o_no_tectonic']].dropna()

ls = LombScargle(data['age_final'], data['d18o_no_tectonic'])
frequency_notec2, power_notec2 = ls.autopower(minimum_frequency=0.001,  # periods < 2 years
                                 maximum_frequency=12,   # periods > 1 month
                                 samples_per_peak=10)
period_notec = 1 / frequency_notec



data = df_sorted[['age_final', 'd18o_no_tectonic2']].dropna()

ls = LombScargle(data['age_final'], data['d18o_no_tectonic2'])
frequency_notec2, power_notec2 = ls.autopower(minimum_frequency=0.001,  # periods < 2 years
                                 maximum_frequency=12,   # periods > 1 month
                                 samples_per_peak=10)
period_notec2 = 1 / frequency_notec2



In [ ]:
fig, ax = plt.subplots(1, figsize=(8, 5))

blues = cm.get_cmap('Blues')

colors = [
    blues(0.35),  # light green (raw)
    blues(0.65),  # medium green (36m)
    blues(0.9)    # dark green (72m)
]

ax.plot(period, power,color=colors[0], label='raw')
ax.plot(period_notec, power_notec,color=colors[1], label='36 month filter')
ax.plot(period_notec2, power_notec2,color=colors[2], label='72 month filter')


ax.axvline(1.0, color='r', linestyle='--', alpha=0.4,label='1 year')
ax.axvline(3.617, color='g', linestyle='--', alpha=0.4,label='3.6 year')
ax.set_xlabel('Period (years)')
ax.set_ylabel('Spectral Amplitude')
ax.set_title(r'$\delta^{13}$C Spectrum')
ax.set_xlim(0.5, 7)
ax.set_ylim(0,0.15)

ax.legend(loc='upper right')

fig.savefig('lombscargle_raw-filter_d18o.pdf')

In [ ]:
data = enso[['decimal_year', 'ENSO']].dropna()

ls = LombScargle(data['decimal_year'], data['ENSO'])
frequency_enso, power_enso = ls.autopower(minimum_frequency=0.001,  # periods < 2 years
                                 maximum_frequency=12,   # periods > 1 month
                                 samples_per_peak=10)
period_enso = 1 / frequency_enso

data = climate[['age_absolute', 'synthetic_d18o']].dropna()

ls = LombScargle(data['age_absolute'], data['synthetic_d18o'])
frequency_synd18o, power_synd18o = ls.autopower(minimum_frequency=0.001,  # periods < 2 years
                                 maximum_frequency=12,   # periods > 1 month
                                 samples_per_peak=10)
period_synd18o = 1 / frequency_synd18o



data = climate[['age_absolute', 'SSS']].dropna()

ls = LombScargle(data['age_absolute'], data['SSS'])
frequency_SSS, power_SSS = ls.autopower(minimum_frequency=0.001,  # periods < 2 years
                                 maximum_frequency=12,   # periods > 1 month
                                 samples_per_peak=10)
period_SSS = 1 / frequency_SSS



In [ ]:
fig, axes = plt.subplots(3,1, figsize=(8, 9), sharex=True, gridspec_kw={'hspace':0})

periods = [period_enso, period_synd18o, period_SSS]
powers = [power_enso, power_synd18o, power_SSS]
colors = ['#4F4948', '#0827FF', '#E30EDE']
labels = ['ENSO', r'Climate $\delta^{18}$O', 'SSS']


for i in range(3):
    
    ax = axes[i]
    ax.plot(periods[i], powers[i], colors[i], label=labels[i])
    ax.axvline(1.0, color='r', linestyle='--', alpha=0.4,label='1 year')
    ax.axvline(3.617, color='g', linestyle='--', alpha=0.4,label='3.6 year')
    ax.set_xlabel('Period (years)')
    ax.set_ylabel('Spectral Amplitude')
    #ax.set_title(r'$\delta^{13}$C Spectrum')
    ax.set_xlim(0.5, 7)
    ax.set_ylim(0,0.15)
    ax.legend(loc='upper right')

#axes[1].set_ylim(0, 0.2)
#axes[2].set_ylim(0, 0.2)
ax.legend(loc='upper right')

fig.savefig('spectral_enso-d18o-sss.pdf')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from astropy.timeseries import LombScargle

def lomb_spec(df, xcol, ycol, fmin=0.001, fmax=12, spp=10):
    d = df[[xcol, ycol]].dropna()
    ls = LombScargle(d[xcol].values, d[ycol].values)
    freq, power = ls.autopower(minimum_frequency=fmin,
                               maximum_frequency=fmax,
                               samples_per_peak=spp)
    period = 1.0 / freq
    return period, power

# ---- compute spectra ----
# d13c
p_d13c_raw,  pow_d13c_raw  = lomb_spec(df_sorted, 'age_final', 'd13c')
p_d13c_36,   pow_d13c_36   = lomb_spec(df_sorted, 'age_final', 'd13c_no_tectonic')
p_d13c_72,   pow_d13c_72   = lomb_spec(df_sorted, 'age_final', 'd13c_no_tectonic2')

# d18o
p_d18o_raw,  pow_d18o_raw  = lomb_spec(df_sorted, 'age_final', 'd18o')
p_d18o_36,   pow_d18o_36   = lomb_spec(df_sorted, 'age_final', 'd18o_no_tectonic')
p_d18o_72,   pow_d18o_72   = lomb_spec(df_sorted, 'age_final', 'd18o_no_tectonic2')

# ---- colors ----
greens = cm.get_cmap('Greens')
blues  = cm.get_cmap('Blues')

c_raw_g, c_36_g, c_72_g = greens(0.35), greens(0.65), greens(0.90)
c_raw_b, c_36_b, c_72_b = blues(0.35),  blues(0.65),  blues(0.90)

# ---- plot ----
fig, axes = plt.subplots(2, 1, figsize=(8, 8), sharex=True,
                         gridspec_kw={'hspace': 0})

# panel 1: d13c
ax = axes[0]
ax.plot(p_d13c_raw, pow_d13c_raw, color=c_raw_g, lw=1.4, label='Raw')
ax.plot(p_d13c_36,  pow_d13c_36,  color=c_36_g,  lw=1.6, label='36-month filter')
ax.plot(p_d13c_72,  pow_d13c_72,  color=c_72_g,  lw=1.8, label='72-month filter')
ax.axvline(1.0,   color='gray', linestyle='--', alpha=0.6, lw=1.0)
ax.axvline(3.617, color='gray', linestyle='--', alpha=0.6, lw=1.0)
ax.set_ylabel(r'$\delta^{13}$C Spectrum'+'\n'+'Spectral Amplitude')
#ax.set_title(r'$\delta^{13}$C Spectrum')
ax.set_xlim(0.5, 7)
ax.set_ylim(0, 0.04)
ax.legend(loc='upper right', frameon=False)
ax.grid(alpha=0.15)

# panel 2: d18o
ax = axes[1]
ax.plot(p_d18o_raw, pow_d18o_raw, color=c_raw_b, lw=1.4, label='Raw')
ax.plot(p_d18o_36,  pow_d18o_36,  color=c_36_b,  lw=1.6, label='36-month filter')
ax.plot(p_d18o_72,  pow_d18o_72,  color=c_72_b,  lw=1.8, label='72-month filter')
ax.axvline(1.0,   color='gray', linestyle='--', alpha=0.6, lw=1.0, label='1 year')
ax.axvline(3.617, color='gray', linestyle='--', alpha=0.6, lw=1.0, label='3.6 years')
ax.set_xlabel('Period (years)')
ax.set_ylabel(r'$\delta^{18}$O Spectrum'+'\n'+'Spectral Amplitude')
#ax.set_title(r'$\delta^{18}$O Spectrum')
ax.set_xlim(0.5, 7)
ax.set_ylim(0, 0.2)
ax.legend(loc='upper right', frameon=False)
ax.grid(alpha=0.15)

plt.tight_layout()
fig.savefig('spectral_d13c_d18o_raw-36m-72m.pdf')
plt.show()